<a href="https://colab.research.google.com/github/Kazi-Sadman/General-Track/blob/main/Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# =============================================================================
# THESIS: An Explainable Language-Aware Cross-Lingual Framework for
#         Bangla-English Fake News Detection Using Multilingual Transformers
#         and Classical Machine Learning
#
# FILE: Single-file end-to-end implementation for Google Colab / Jupyter
# SEED: 42  |  TEST SET: frozen until Section 26 (Final Evaluation)
# ===================================================


# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================
# Installs all dependencies needed for the full pipeline.
# Safe to re-run (pip skips already-installed packages).

In [1]:

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn \
    transformers datasets accelerate sentencepiece langdetect lime \
    joblib tqdm pyyaml openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 27.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# SECTION 2 — IMPORTS
# =============================================================================
# Import every library used later. Names are grouped by purpose.

# --- Standard library ---

In [2]:
import os
import re
import sys
import json
import time
import html
import random
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

# --- Data handling ---

In [3]:
import numpy as np
import pandas as pd

# --- Visualisation ---

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Classical machine learning ---

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_fscore_support
)
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier

# --- Language identification ---

In [6]:
from langdetect import detect as detect_language_langdetect
from langdetect import DetectorFactory, LangDetectException
DetectorFactory.seed = 0

# --- Explainability ---

In [7]:
from lime.lime_text import LimeTextExplainer


# --- Deep learning (transformers) --

In [8]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)